# Part 2: Named Entity Recognition using Neural Network

This notebook implements a Feed-Forward Neural Network for Named Entity Recognition using the Word2Vec embeddings learned in Part 1.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from datasets import load_dataset
import pickle
import json
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cpu


## 1. Load Word Embeddings from Part 1


In [3]:
embeddings = np.load('word_embeddings.npy')

with open('word2idx.json', 'r') as f:
    word2idx_raw = json.load(f)

word2idx = {word: int(idx) for word, idx in word2idx_raw.items()}
idx2word = {idx: word for word, idx in word2idx.items()}

print(f"Loaded embeddings: {embeddings.shape}")
print(f"Vocabulary size: {len(word2idx)}")
print(f"Embedding dimension: {embeddings.shape[1]}")

vocab_size = len(word2idx)
embedding_dim = embeddings.shape[1]

PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

if PAD_TOKEN not in word2idx:
    word2idx[PAD_TOKEN] = vocab_size
    idx2word[vocab_size] = PAD_TOKEN
    pad_embedding = np.zeros(embedding_dim)
    embeddings = np.vstack([embeddings, pad_embedding])
    vocab_size += 1

if UNK_TOKEN not in word2idx:
    word2idx[UNK_TOKEN] = vocab_size
    idx2word[vocab_size] = UNK_TOKEN
    vocab_size += 1
    unk_embedding = np.random.normal(0, 0.01, embedding_dim)
    embeddings = np.vstack([embeddings, unk_embedding])

PAD_IDX = word2idx[PAD_TOKEN]
UNK_IDX = word2idx[UNK_TOKEN]

print(f"Final vocab size: {vocab_size}")
print(f"Final embeddings shape: {embeddings.shape}")
print(f"PAD token index: {PAD_IDX}")
print(f"UNK token index: {UNK_IDX}")


Loaded embeddings: (6273, 100)
Vocabulary size: 6273
Embedding dimension: 100
Final vocab size: 6275
Final embeddings shape: (6275, 100)
PAD token index: 6273
UNK token index: 6274


## 2. Load and Prepare NER Dataset


In [ ]:
dataset = load_dataset('lhoestq/conll2003')

print("Dataset loaded:")
print(f"  Train: {len(dataset['train'])} samples")
print(f"  Validation: {len(dataset['validation'])} samples")
print(f"  Test: {len(dataset['test'])} samples")

example = dataset['train'][0]
print(f"\nExample sample:")
print(f"  Tokens: {example['tokens']}")
print(f"  NER tags: {example['ner_tags']}")

print(f"\nDataset features structure:")
print(f"  ner_tags feature type: {type(dataset['train'].features['ner_tags'])}")
print(f"  ner_tags feature: {dataset['train'].features['ner_tags']}")


dataset_infos.json: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

Dataset loaded:
  Train: 14041 samples
  Validation: 3250 samples
  Test: 3453 samples

Example sample:
  Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
  NER tags: [3, 0, 7, 0, 0, 0, 7, 0, 0]


In [6]:
try:
    ner_feature = dataset['train'].features['ner_tags']
    if hasattr(ner_feature, 'feature') and hasattr(ner_feature.feature, 'names'):
        tag_names = ner_feature.feature.names
    elif hasattr(ner_feature, 'names'):
        tag_names = ner_feature.names
    else:
        raise AttributeError("Cannot find names attribute")
except (AttributeError, KeyError):
    tag_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

print("NER Tag Names:")
for idx, tag in enumerate(tag_names):
    print(f"  {idx}: {tag}")

num_labels = len(tag_names)
print(f"\nTotal number of NER labels: {num_labels}")

tag2idx = {tag: idx for idx, tag in enumerate(tag_names)}
idx2tag = {idx: tag for idx, tag in enumerate(tag_names)}


NER Tag Names:
  0: O
  1: B-PER
  2: I-PER
  3: B-ORG
  4: I-ORG
  5: B-LOC
  6: I-LOC
  7: B-MISC
  8: I-MISC

Total number of NER labels: 9


## 3. Prepare Training Data


In [7]:
def get_word_index(word):
    word_lower = word.lower()
    if word_lower in word2idx:
        return word2idx[word_lower]
    else:
        return UNK_IDX

def prepare_data(split_data):
    X = []
    y = []
    
    for sample in split_data:
        tokens = sample['tokens']
        ner_tags = sample['ner_tags']
        
        word_indices = [get_word_index(token) for token in tokens]
        X.append(word_indices)
        y.append(ner_tags)
    
    return X, y

X_train, y_train = prepare_data(dataset['train'])
X_val, y_val = prepare_data(dataset['validation'])
X_test, y_test = prepare_data(dataset['test'])

print(f"Train samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")
print(f"\nExample:")
print(f"  Tokens: {dataset['train'][0]['tokens']}")
print(f"  Word indices: {X_train[0]}")
print(f"  NER tags: {y_train[0]}")


Train samples: 14041
Validation samples: 3250
Test samples: 3453

Example:
  Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
  Word indices: [0, 6274, 1, 2, 3, 4, 5, 6274, 6]
  NER tags: [3, 0, 7, 0, 0, 0, 7, 0, 0]


## 4. Create Dataset and DataLoader


In [8]:
class NERDataset(Dataset):
    def __init__(self, X, y, max_length=None):
        self.X = X
        self.y = y
        self.max_length = max_length
        
        if max_length is None:
            self.max_length = max(len(seq) for seq in X)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        seq = self.X[idx]
        tags = self.y[idx]
        
        if len(seq) > self.max_length:
            seq = seq[:self.max_length]
            tags = tags[:self.max_length]
        
        padded_seq = seq + [PAD_IDX] * (self.max_length - len(seq))
        padded_tags = tags + [-1] * (self.max_length - len(tags))
        mask = [1] * len(self.X[idx]) + [0] * (self.max_length - len(self.X[idx]))
        
        return {
            'input_ids': torch.tensor(padded_seq, dtype=torch.long),
            'labels': torch.tensor(padded_tags, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.bool)
        }

max_length = 128

train_dataset = NERDataset(X_train, y_train, max_length=max_length)
val_dataset = NERDataset(X_val, y_val, max_length=max_length)
test_dataset = NERDataset(X_test, y_test, max_length=max_length)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


Train batches: 220
Validation batches: 51
Test batches: 54


## 5. Define Feed-Forward Neural Network Model


In [9]:
class FeedForwardNER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_labels, hidden_dim=256, num_layers=2, dropout=0.3, embeddings=None):
        super(FeedForwardNER, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.num_labels = num_labels
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        if embeddings is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(embeddings))
            self.embedding.weight.requires_grad = False
        
        layers = []
        input_dim = embedding_dim
        
        for i in range(num_layers):
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            input_dim = hidden_dim
        
        self.feedforward = nn.Sequential(*layers)
        
        self.classifier = nn.Linear(hidden_dim, num_labels)
    
    def forward(self, input_ids, mask=None):
        embedded = self.embedding(input_ids)
        
        batch_size, seq_len, emb_dim = embedded.shape
        
        embedded_flat = embedded.view(-1, emb_dim)
        
        output = self.feedforward(embedded_flat)
        
        logits = self.classifier(output)
        
        logits = logits.view(batch_size, seq_len, self.num_labels)
        
        return logits

model = FeedForwardNER(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_labels=num_labels,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3,
    embeddings=embeddings
)

model = model.to(device)

print("Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Model architecture:
FeedForwardNER(
  (embedding): Embedding(6275, 100)
  (feedforward): Sequential(
    (0): Linear(in_features=100, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=256, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
  )
  (classifier): Linear(in_features=256, out_features=9, bias=True)
)

Total parameters: 721,461
Trainable parameters: 93,961


## 6. Training Configuration


In [10]:
criterion = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

num_epochs = 20
print(f"Training configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: 0.001")
print(f"  Hidden dimension: 256")
print(f"  Number of layers: 2")
print(f"  Dropout: 0.3")


Training configuration:
  Epochs: 20
  Batch size: 64
  Learning rate: 0.001
  Hidden dimension: 256
  Number of layers: 2
  Dropout: 0.3


## 7. Training Function


In [11]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        mask = batch['mask'].to(device)
        
        optimizer.zero_grad()
        
        logits = model(input_ids, mask)
        
        logits_flat = logits.view(-1, num_labels)
        labels_flat = labels.view(-1)
        
        loss = criterion(logits_flat, labels_flat)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


## 8. Validation Function


In [12]:
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            mask = batch['mask'].to(device)
            
            logits = model(input_ids, mask)
            
            logits_flat = logits.view(-1, num_labels)
            labels_flat = labels.view(-1)
            
            loss = criterion(logits_flat, labels_flat)
            total_loss += loss.item()
            
            predictions = torch.argmax(logits, dim=-1)
            
            for i in range(predictions.size(0)):
                seq_len = mask[i].sum().item()
                valid_labels = labels[i][:seq_len].cpu().numpy()
                valid_preds = predictions[i][:seq_len].cpu().numpy()
                valid_mask = valid_labels != -1
                all_predictions.extend(valid_preds[valid_mask].tolist())
                all_labels.extend(valid_labels[valid_mask].tolist())
    
    avg_loss = total_loss / len(dataloader)
    return avg_loss, all_predictions, all_labels


## 9. Training Loop


In [13]:
train_losses = []
val_losses = []
val_f1_scores = []

best_f1 = 0.0
patience_counter = 0
patience = 5

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    print("-" * 50)
    
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_preds, val_labels = validate(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted', zero_division=0
    )
    accuracy = accuracy_score(val_labels, val_preds)
    
    val_f1_scores.append(f1)
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Accuracy: {accuracy:.4f}")
    print(f"Val Precision: {precision:.4f}")
    print(f"Val Recall: {recall:.4f}")
    print(f"Val F1-Score: {f1:.4f}")
    
    scheduler.step(val_loss)
    
    if f1 > best_f1:
        best_f1 = f1
        patience_counter = 0
        torch.save(model.state_dict(), 'best_ner_model.pt')
        print("Saved best model!")
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping triggered after {epoch + 1} epochs")
        break

print(f"\nTraining completed. Best validation F1: {best_f1:.4f}")



Epoch 1/20
--------------------------------------------------


Validating: 100%|██████████| 51/51 [00:01<00:00, 38.60it/s]


Train Loss: 0.5976
Val Loss: 0.4086
Val Accuracy: 0.8863
Val Precision: 0.8738
Val Recall: 0.8863
Val F1-Score: 0.8528
Saved best model!

Epoch 2/20
--------------------------------------------------


Validating: 100%|██████████| 51/51 [00:01<00:00, 39.92it/s]


Train Loss: 0.3770
Val Loss: 0.3530
Val Accuracy: 0.9025
Val Precision: 0.8856
Val Recall: 0.9025
Val F1-Score: 0.8806
Saved best model!

Epoch 3/20
--------------------------------------------------


...

Train Loss: 0.2701
Val Loss: 0.3064
Val Accuracy: 0.9171
Val Precision: 0.9059
Val Recall: 0.9171
Val F1-Score: 0.9048
Saved best model!

Epoch 19/20
--------------------------------------------------


Validating: 100%|██████████| 51/51 [00:01<00:00, 28.61it/s]


Train Loss: 0.2693
Val Loss: 0.3074
Val Accuracy: 0.9169
Val Precision: 0.9056
Val Recall: 0.9169
Val F1-Score: 0.9040

Epoch 20/20
--------------------------------------------------


Validating: 100%|██████████| 51/51 [00:01<00:00, 31.31it/s]


Train Loss: 0.2685
Val Loss: 0.3062
Val Accuracy: 0.9166
Val Precision: 0.9045
Val Recall: 0.9166
Val F1-Score: 0.9043

Training completed. Best validation F1: 0.9048


## 10. Load Best Model and Evaluate on Test Set


In [14]:
model.load_state_dict(torch.load('best_ner_model.pt'))
print("Loaded best model for testing")

test_loss, test_preds, test_labels = validate(model, test_loader, criterion, device)

print(f"\nTest Loss: {test_loss:.4f}")

accuracy = accuracy_score(test_labels, test_preds)
precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_preds, average='weighted', zero_division=0
)

print(f"\nTest Set Evaluation:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    test_labels, test_preds, average='macro', zero_division=0
)

print(f"\nMacro-averaged metrics:")
print(f"  Precision: {precision_macro:.4f}")
print(f"  Recall: {recall_macro:.4f}")
print(f"  F1-Score: {f1_macro:.4f}")


Loaded best model for testing


Validating: 100%|██████████| 54/54 [00:01<00:00, 34.37it/s]



Test Loss: 0.3846

Test Set Evaluation:
  Accuracy: 0.8990
  Precision: 0.8825
  Recall: 0.8990
  F1-Score: 0.8822

Macro-averaged metrics:
  Precision: 0.7212
  Recall: 0.5036
  F1-Score: 0.5772


## 11. Detailed Classification Report


In [15]:
print("Detailed Classification Report:")
print(classification_report(
    test_labels, 
    test_preds, 
    target_names=tag_names,
    zero_division=0
))


Detailed Classification Report:
              precision    recall  f1-score   support

           O       0.92      0.99      0.95     38323
       B-PER       0.79      0.41      0.54      1617
       I-PER       0.49      0.13      0.21      1156
       B-ORG       0.74      0.47      0.57      1661
       I-ORG       0.65      0.30      0.41       835
       B-LOC       0.79      0.74      0.76      1668
       I-LOC       0.66      0.50      0.57       257
      B-MISC       0.76      0.56      0.64       702
      I-MISC       0.69      0.44      0.53       216

    accuracy                           0.90     46435
   macro avg       0.72      0.50      0.58     46435
weighted avg       0.88      0.90      0.88     46435



## 12. Per-Class Metrics


In [16]:
precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
    test_labels, test_preds, labels=range(num_labels), zero_division=0
)

print("Per-Class Metrics:")
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-" * 70)
for i in range(num_labels):
    print(f"{tag_names[i]:<15} {precision_per_class[i]:<12.4f} {recall_per_class[i]:<12.4f} {f1_per_class[i]:<12.4f} {support_per_class[i]:<10}")


Per-Class Metrics:
Class           Precision    Recall       F1-Score     Support   
----------------------------------------------------------------------
O               0.9184       0.9931       0.9543       38323     
B-PER           0.7919       0.4094       0.5397       1617      
I-PER           0.4859       0.1341       0.2102       1156      
B-ORG           0.7383       0.4654       0.5709       1661      
I-ORG           0.6527       0.2994       0.4105       835       
B-LOC           0.7875       0.7422       0.7642       1668      
I-LOC           0.6598       0.4981       0.5676       257       
B-MISC          0.7647       0.5556       0.6436       702       
I-MISC          0.6912       0.4352       0.5341       216       


## 13. Example Predictions


In [17]:
model.eval()
with torch.no_grad():
    sample_idx = 0
    sample = dataset['test'][sample_idx]
    
    tokens = sample['tokens']
    true_tags = sample['ner_tags']
    
    word_indices = [get_word_index(token) for token in tokens]
    
    if len(word_indices) > max_length:
        word_indices = word_indices[:max_length]
        true_tags = true_tags[:max_length]
    
    padded_seq = word_indices + [PAD_IDX] * (max_length - len(word_indices))
    
    input_tensor = torch.tensor([padded_seq], dtype=torch.long).to(device)
    
    logits = model(input_tensor)
    predictions = torch.argmax(logits, dim=-1)[0]
    
    pred_tags = predictions[:len(tokens)].cpu().numpy().tolist()
    
    print("Example Prediction:")
    print(f"Tokens: {tokens}")
    print(f"\nTrue Tags:")
    print(f"  {[tag_names[t] for t in true_tags]}")
    print(f"\nPredicted Tags:")
    print(f"  {[tag_names[p] for p in pred_tags]}")
    
    print(f"\nCorrect predictions: {sum(p == t for p, t in zip(pred_tags, true_tags))}/{len(tokens)}")


Example Prediction:
Tokens: ['SOCCER', '-', 'JAPAN', 'GET', 'LUCKY', 'WIN', ',', 'CHINA', 'IN', 'SURPRISE', 'DEFEAT', '.']

True Tags:
  ['O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'B-PER', 'O', 'O', 'O', 'O']

Predicted Tags:
  ['O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O']

Correct predictions: 11/12
